# Scalar wave propagation: from Helmholtz to angular spectrum

**The central question:** if the complex scalar field is known on one plane, how do we find it on another?

**Medium and wavelength convention.** The surrounding medium is homogeneous, isotropic and lossless.

- $\lambda$ is wavelength **in the medium**: $\lambda=\lambda_0/n$ and $k=nk_0=2\pi/\lambda$.
- The controls vary medium wavelength; there is no separate index control.

See the [course medium convention](../0_0_Foundations_of_Scalar_Optics.ipynb#medium-and-wavelength).

This notebook organizes *Scalar_wave_propagation.docx* into one derivation. It retains the document's field, Fourier, wavevector, spatial-frequency, direction-cosine, angular, and Green-function representations, together with all twelve original figures. The two source animations remain animated. Repeated explanations are combined; mathematical corrections are recorded at the end.

The [scalar-optics foundations notebook](../0_0_Foundations_of_Scalar_Optics.ipynb) introduces the wave equation and its solutions. Here we use that foundation to derive propagation. No earlier notebook needs to be run.

**Reading route**

1. [The scalar field and Helmholtz equation](#1.-The-field-we-propagate)
2. [A field on a plane and its Fourier spectrum](#2.-The-input-plane-and-its-Fourier-spectrum)
3. [Spatial frequency becomes a direction](#3.-One-component,-several-coordinate-descriptions)
4. [Deriving the angular spectrum method](#4.-Helmholtz-becomes-an-ordinary-differential-equation)
5. [Propagating and evanescent components](#5.-The-exact-angular-spectrum-propagator)
6. [The Fresnel approximation](#6.-Fresnel:-approximating-the-longitudinal-wavevector)
7. [How much phase error is acceptable?](#7.-The-error-is-an-accumulated-phase-error)
8. [Convolution and Rayleigh–Sommerfeld](#8.-The-same-propagation-in-real-space)
9. [A small computation](#9.-A-first-FFT-calculation)

The optional sections retain the document's additional representations without interrupting this route. Throughout, wavelength is the wavelength **in the medium**, propagation is into $z>0$, and physical time dependence is $e^{-i\omega t}$.

In [ ]:
# Find the shared wave_optics package in the project root.
from pathlib import Path
import sys
import importlib

_support_dir = next((base
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (base / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
# Relocate a cached package without changing the lesson's module reload order.
importlib.invalidate_caches()
if 'wave_optics' in sys.modules:
    sys.modules['wave_optics'].__path__[:] = [str(_support_dir / 'wave_optics')]

from pathlib import Path
import importlib
import numpy as np
from IPython.display import display, Image
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

# Rerunning setup releases earlier views before reloading their definitions.
for lesson in globals().get('scalar_propagation_lessons', {}).values():
    lesson.close()
scalar_propagation_lessons = {}

import wave_optics.scalar_propagation as propagation
import wave_optics.scalar_propagation_view as propagation_view
importlib.reload(propagation)
importlib.reload(propagation_view)
from wave_optics.plotting import figure_png

def show_scalar_propagation(stage):
    if stage in scalar_propagation_lessons:
        scalar_propagation_lessons[stage].close()
    lesson = propagation_view.ScalarPropagationExplorer(stage)
    scalar_propagation_lessons[stage] = lesson
    display(lesson.ui)
    return lesson

def show_figure(figure):
    display(Image(data=figure_png(figure)))

## 1. The field we propagate

### 1.1 From the wave equation to Helmholtz

In a source-free, homogeneous, isotropic, linear medium with constant $\epsilon$ and $\mu$, Maxwell's equations give

$$
\nabla\times\mathbf E=-\mu\,\partial_t\mathbf H,
\qquad
\nabla\times\mathbf H=\epsilon\,\partial_t\mathbf E,
\qquad \nabla\cdot\mathbf E=0.
$$

Taking one more curl and using $\nabla\times(\nabla\times\mathbf E)=\nabla(\nabla\cdot\mathbf E)-\nabla^2\mathbf E$ gives

$$
-\nabla^2\mathbf E=-\mu\epsilon\,\partial_t^2\mathbf E
\quad\Longrightarrow\quad
\nabla^2\mathbf E-\frac{1}{v^2}\partial_t^2\mathbf E=0,
\qquad v=\frac{1}{\sqrt{\mu\epsilon}}.
$$

Each Cartesian component obeys this scalar wave equation. We describe one monochromatic scalar field by

$$u(\mathbf r,t)=\operatorname{Re}\{U(\mathbf r)e^{-i\omega t}\}.$$

Since $\partial_t^2e^{-i\omega t}=-\omega^2e^{-i\omega t}$, substitution yields

$$
\boxed{(\nabla^2+k^2)U=0},
\qquad k=\frac{\omega}{v}=\frac{2\pi}{\lambda}.
$$

This is the **scalar Helmholtz equation**. A scalar model describes spatial amplitude and phase; it does not by itself enforce all vector Maxwell constraints. We use it throughout this notebook, in a uniform lossless medium.

### 1.2 Three ways to describe the same complex field

$$
U=U_R+iU_I=A_{\mathrm{amp}}e^{i\phi},
\qquad
A_{\mathrm{amp}}=|U|=\sqrt{U_R^2+U_I^2},
\qquad \phi=\arg U.
$$

Therefore

$$u(\mathbf r,t)=A_{\mathrm{amp}}(\mathbf r)
\cos[\phi(\mathbf r)-\omega t].$$

The phasor $U$ contains the local amplitude and phase, but omits the common time oscillation. It need not vary slowly in space. A separate carrier–envelope representation is

$$U(x,y,z)=\psi(x,y,z)e^{ikz};$$

only in a paraxial approximation will we assume the envelope $\psi$ varies slowly along $z$.

For the plots we use **relative scalar intensity** $I_{\mathrm{rel}}=|U|^2$ with unit input amplitude as the reference. An electric-field phasor has units V/m; an irradiance-normalized scalar amplitude can instead have units $\sqrt{\mathrm W}/\mathrm m$, giving $|U|^2$ in W/m². These are different normalizations. A full vector calculation is needed to identify energy flux for arbitrary directions or evanescent fields.

### 1.3 Why superposition helps

For constants $a,b$, linearity gives

$$
(\nabla^2+k^2)(aU_1+bU_2)
=a(\nabla^2+k^2)U_1+b(\nabla^2+k^2)U_2=0.
$$

We can build a complicated field from simpler solutions. The next step is to choose a set whose propagation is easy: plane-wave components.

## 2. The input plane and its Fourier spectrum

### 2.1 What is known, and what is unknown?

We prescribe the complex field on $z=0$,

$$U_{\mathrm{ref}}(\xi,\eta)=U(\xi,\eta,0),$$

and seek $U(x,y,z)$ in the homogeneous half-space $z>0$. The source coordinates $(\xi,\eta)$ and observation coordinates $(x,y)$ distinguish two planes; they do not introduce different kinds of position.

An aperture model might supply $U_{\mathrm{ref}}=t\,U_i$. Propagation starts once this boundary field is given; solving the electromagnetic scattering at an actual aperture is a separate problem. An outgoing condition will also be needed to select the field in $z>0$.

If this input plane is the physical exit, $U_{\mathrm{ref}}=U_{\mathrm{exit}}$. Otherwise it is a separate reference plane, reached by propagating the exit field. The notation distinguishes the planes while describing the same underlying optical solution.

**Notation:** use $U_{\mathrm{exit}}$ at the exit plane, $U_{\mathrm{ref}}$ at a general reference plane, and $C_m$ (or $C_{mn}$) for field coefficients. [Shared convention](../0_0_Foundations_of_Scalar_Optics.ipynb#exit-field-notation).


<img src="figures/scalar_wave_propagation_source/image2.jpeg" width="640" alt="Source figure: an aperture/source plane and an observation point. The source uses $P_1$ and $P_0$ for the two points; the integration variable belongs to the source plane." />

Source figure: an aperture/source plane and an observation point. The source uses $P_1$ and $P_0$ for the two points; the integration variable belongs to the source plane.

### 2.2 The two-dimensional Fourier pair

Let $f_x,f_y$ be transverse spatial frequencies in **cycles per unit length**. Write

$$
\boxed{\widetilde U_{\mathrm{ref}}(f_x,f_y)=\iint U_{\mathrm{ref}}(\xi,\eta)
e^{-i2\pi(f_x\xi+f_y\eta)}\,d\xi\,d\eta}
$$

and

$$
\boxed{U_{\mathrm{ref}}(\xi,\eta)=\iint \widetilde U_{\mathrm{ref}}(f_x,f_y)
e^{i2\pi(f_x\xi+f_y\eta)}\,df_x\,df_y.}
$$

The spectrum $\widetilde U$ gives the complex weight of each sinusoidal component. It is not merely an intensity spectrum: its phase is essential when we add the components back together. Here $\widetilde U(f_x,f_y;z)$ denotes a spectrum, while $A_{\mathrm{amp}}(\mathbf r)$ in Section 1 denotes a local field magnitude.

We transform only $x,y$ because the given boundary field has two spatial coordinates. We leave $z$ as the propagation variable. A full 3D Fourier representation is also possible under suitable assumptions; its role is discussed in the optional section.

Finite-transverse-power fields admit an $L^2$ Fourier representation. Hard apertures need not be differentiable. Ideal infinite plane waves use delta distributions rather than ordinary integrable spectra.

<img src="figures/scalar_wave_propagation_source/image4.png" width="700" alt="Source figure: three transverse sinusoidal components and their sum. These signed colour scales represent field values, not intensity. The original EMF is retained alongside this PNG display copy." />

Source figure: three transverse sinusoidal components and their sum. These signed colour scales represent field values, not intensity. The original EMF is retained alongside this PNG display copy.

<img src="figures/scalar_wave_propagation_source/image3.png" width="1000" alt="Source figure: decomposition, independent propagation, and recombination. Its field label $E$ corresponds to our scalar $U$. Each component will acquire its own factor $e^{ik_z z}$. The component panels depict field/phase variation, not separate intensity beams." />

Source figure: decomposition, independent propagation, and recombination. Its field label $E$ corresponds to our scalar $U$. Each component will acquire its own factor $e^{ik_z z}$. The component panels depict field/phase variation, not separate intensity beams.

## 3. One component, several coordinate descriptions

### 3.1 Matching a Fourier component to a plane wave

A plane wave has spatial phasor

$$U_{\mathbf k}(x,y,z)=a\,e^{i(k_xx+k_yy+k_zz)}.$$

On $z=0$, comparison with the Fourier basis gives

$$k_x=2\pi f_x,\qquad k_y=2\pi f_y.$$

Substituting the plane wave into Helmholtz gives

$$
\nabla^2U_{\mathbf k}=-(k_x^2+k_y^2+k_z^2)U_{\mathbf k}
\quad\Longrightarrow\quad
k_x^2+k_y^2+k_z^2=k^2.
$$

The transverse frequencies therefore determine the longitudinal component up to a branch choice. For forward propagation we will use

$$k_z=+\sqrt{k^2-k_x^2-k_y^2}.$$

### 3.2 The coordinate dictionary

For a **propagating** component, write $\mathbf k=k\hat{\mathbf s}$. The same direction can be described as follows:

| Representation | Components | Constraint |
|:--|:--|:--|
| Wavevector, radians/length | $(k_x,k_y,k_z)$ | $k_x^2+k_y^2+k_z^2=k^2$ |
| Spatial frequency, cycles/length | $(f_x,f_y,f_z)=\mathbf k/(2\pi)$ | $f_x^2+f_y^2+f_z^2=1/\lambda^2$ |
| Direction cosines, dimensionless | $(\alpha,\beta,\gamma)=\mathbf k/k$ | $\alpha^2+\beta^2+\gamma^2=1$ |
| Polar and azimuthal angles | $(\theta,\varphi)$ | $\hat{\mathbf s}=(\sin\theta\cos\varphi,\sin\theta\sin\varphi,\cos\theta)$ |
| Angles to the three axes | $(\theta_x,\theta_y,\theta_z)$ | $(\alpha,\beta,\gamma)=(\cos\theta_x,\cos\theta_y,\cos\theta_z)$ |

Thus

$$
\alpha=\lambda f_x,\quad \beta=\lambda f_y,\quad
\gamma=\lambda f_z=\sqrt{1-\alpha^2-\beta^2},
$$

$$
\theta=\theta_z=\arccos\gamma
=\arcsin\sqrt{\alpha^2+\beta^2},
\qquad \varphi=\operatorname{atan2}(\beta,\alpha).
$$

These inverse-angle formulas select the forward hemisphere, $0\leq\theta\leq\pi/2$. At $\alpha=\beta=0$ the azimuth is undefined. Evanescent components do not have an ordinary real propagation angle.

The source also uses $(a,b,c)$ for spatial frequencies: $a=f_x$, $b=f_y$, $c=f_z$. In that notation, **$c$ is not the speed of light**.

<img src="figures/scalar_wave_propagation_source/image5.png" width="300" alt="Notation guide for the retained figure: its $\alpha,\beta,\gamma$ label **axis angles**. In this notebook those angles are $\theta_x,\theta_y,\theta_z$; their cosines are $\alpha,\beta,\gamma$. The direction arrow $u$ represents $\hat{\mathbf s}$, parallel to $\mathbf k$ (a separate $\mathbf k$ label was overlaid in Word)." />

Notation guide for the retained figure: its $\alpha,\beta,\gamma$ label **axis angles**. In this notebook those angles are $\theta_x,\theta_y,\theta_z$; their cosines are $\alpha,\beta,\gamma$. The direction arrow $u$ represents $\hat{\mathbf s}$, parallel to $\mathbf k$ (a separate $\mathbf k$ label was overlaid in Word).

### 3.3 A component you can move

The left plot locates $(\alpha,\beta)$ in the transverse spectrum; the right shows a signed field snapshot in the physical $xz$ plane at $y=0$ and $t=0$. The arrow is a projection of the propagation direction. It is not a phasor arrow.

Start with $\alpha=0.3$, $\beta=0.4$. Then $\gamma=\sqrt{0.75}$ and $\theta=30^\circ$. Change the two transverse controls separately, then try $(1,0)$ and $(1.2,0)$. The boundary between a propagating and a decaying component is $\alpha^2+\beta^2=1$.

Lengths on the snapshot axes are in wavelength units. Changing wavelength at fixed $\alpha,\beta$ changes the physical scale, not the normalized pattern.

In [ ]:
geometry_lesson = show_scalar_propagation('geometry')

## 4. Helmholtz becomes an ordinary differential equation

At every $z$, write the transverse Fourier expansion

$$U(x,y,z)=\iint \widetilde U(f_x,f_y;z)e^{i2\pi(f_xx+f_yy)}\,df_x\,df_y.$$

The transverse derivatives act on the basis functions:

$$
\partial_x^2 U=\iint -(2\pi f_x)^2\widetilde U\,e^{i2\pi(f_xx+f_yy)}\,df_x\,df_y,
$$

$$
\partial_y^2 U=\iint -(2\pi f_y)^2\widetilde U\,e^{i2\pi(f_xx+f_yy)}\,df_x\,df_y,
$$

whereas the longitudinal derivative acts on the spectrum:

$$
\partial_z^2 U=\iint \frac{\partial^2 \widetilde U}{\partial z^2}
e^{i2\pi(f_xx+f_yy)}\,df_x\,df_y.
$$

Substitution into $(\partial_x^2+\partial_y^2+\partial_z^2+k^2)U=0$ gives

$$
\iint\left[
\frac{\partial^2 \widetilde U}{\partial z^2}
+\bigl(k^2-4\pi^2(f_x^2+f_y^2)\bigr)\widetilde U
\right]e^{i2\pi(f_xx+f_yy)}\,df_x\,df_y=0.
$$

Uniqueness of the Fourier transform makes the bracket zero for each frequency pair:

$$
\boxed{\frac{\partial^2 \widetilde U}{\partial z^2}+k_z^2 \widetilde U=0},
\qquad k_z^2=k^2-4\pi^2(f_x^2+f_y^2).
$$

**This is the main simplification:** a PDE coupling spatial coordinates becomes an independent ODE for each transverse Fourier component. In a uniform medium the components do not exchange amplitude with one another.

For $k_z\ne0$ the ODE has the general solution

$$\widetilde U(z)=C_+e^{ik_z z}+C_-e^{-ik_z z}.$$

Knowing $\widetilde U(0)=C_++C_-$ alone does not determine the two constants. For propagation away from the boundary into $z>0$, we choose the outgoing/decaying solution, $C_-=0$. Therefore

$$\boxed{\widetilde U(f_x,f_y;z)=\widetilde U_{\mathrm{ref}}(f_x,f_y)e^{ik_z z}.}$$

At grazing, $k_z=0$, the general ODE solution is $C_0+C_1z$; bounded continuation of the outgoing solution selects the constant term. The same propagation formula then gives $\widetilde U(z)=\widetilde U(0)$.

## 5. The exact angular spectrum propagator

### 5.1 The square root contains the physics

Use the branch with $\operatorname{Re}k_z\geq0$ and $\operatorname{Im}k_z\geq0$. Define

$$q=\lambda^2(f_x^2+f_y^2)=\alpha^2+\beta^2.$$

| Spectrum location | Longitudinal component | Propagation factor |
|:--|:--|:--|
| $q<1$ | $k_z=k\sqrt{1-q}$, real | $e^{ik_z z}$: phase changes, magnitude stays one |
| $q=1$ | $k_z=0$ | $1$: grazing component |
| $q>1$ | $k_z=i\kappa$, $\kappa=k\sqrt{q-1}$ | $e^{-\kappa z}$: evanescent decay |

The rejected evanescent branch grows as $z\to+\infty$. A near-boundary field may contain high transverse frequencies even though those components cannot propagate as ordinary real-angle plane waves.

<img src="figures/scalar_wave_propagation_source/image6.png" width="350" alt="Source frequency-plane diagram. The outer circle is $f_x^2+f_y^2=1/\lambda^2$. The small dashed Fresnel region is schematic: the acceptable paraxial range also depends on distance and the allowed phase error." />

Source frequency-plane diagram. The outer circle is $f_x^2+f_y^2=1/\lambda^2$. The small dashed Fresnel region is schematic: the acceptable paraxial range also depends on distance and the allowed phase error.

### 5.2 Equivalent expressions worth keeping together

The document's different longitudinal representations are the same quantity:

$$
\begin{aligned}
k_z
&=\sqrt{k^2-k_x^2-k_y^2}\\
&=2\pi\sqrt{\lambda^{-2}-f_x^2-f_y^2}\\
&=\frac{2\pi}{\lambda}\sqrt{1-\lambda^2(f_x^2+f_y^2)}\\
&=k\sqrt{1-\alpha^2-\beta^2}=k\gamma=2\pi f_z.
\end{aligned}
$$

For a propagating component this also equals $k\cos\theta$. The transfer function is consequently

$$
\boxed{
H_{\mathrm{AS}}(f_x,f_y;z)
=e^{ik_z z}
=\exp\!\left[ikz\sqrt{1-\lambda^2(f_x^2+f_y^2)}\right].}
$$

Here $k_z$ is a wavevector component; the product $kz$ means $k\times z$.

Multiplication in the spectrum followed by reconstruction gives the **angular spectrum method (ASM)**:

$$
\boxed{U(x,y,z)=\iint \widetilde U_{\mathrm{ref}}(f_x,f_y)H_{\mathrm{AS}}(f_x,f_y;z)
e^{i2\pi(f_xx+f_yy)}\,df_x\,df_y.}
$$

$$
U_{\mathrm{ref}}\ \xrightarrow{\ \mathcal F_{x,y}\ }\ \widetilde U_{\mathrm{ref}}
\ \xrightarrow{\ \times H_{\mathrm{AS}}\ }\ \widetilde U_z
\ \xrightarrow{\ \mathcal F^{-1}_{x,y}\ }\ U_z.
$$

This is exact for the stated homogeneous scalar half-space problem. A sampled FFT implementation adds separate numerical approximations.

### 5.3 Three spheres, one dispersion relation

The wavevector sphere has radius $k$, the spatial-frequency sphere has radius $1/\lambda$, and the direction sphere has radius one:

$$
(f_x,f_y,f_z)\ \xrightarrow{\times2\pi}\ (k_x,k_y,k_z)
\ \xrightarrow{/k}\ (\alpha,\beta,\gamma).
$$

Forward propagating components occupy the upper hemisphere. Its projection onto the transverse plane is a disk. These spheres live in **frequency/direction space**, not physical space; they are not spherical wavefronts. Evanescent components lie outside the real disk and have imaginary longitudinal coordinates.

<img src="figures/scalar_wave_propagation_source/image7.gif" width="800" alt="Original animation: the frequency hemisphere and its transverse projection. The label $c$ means $f_z$, and $\rho=\lambda\sqrt{f_x^2+f_y^2}$. The final point leaves the propagating disk; its longitudinal component is then imaginary. All 88 frames are retained." />

Original animation: the frequency hemisphere and its transverse projection. The label $c$ means $f_z$, and $\rho=\lambda\sqrt{f_x^2+f_y^2}$. The final point leaves the propagating disk; its longitudinal component is then imaginary. All 88 frames are retained.

<img src="figures/scalar_wave_propagation_source/image9.png" width="400" alt="Source unit-sphere representation. For wavevector geometry, read the printed $x,y,z$ axes as $\alpha,\beta,\gamma$, rather than as physical positions." />

Source unit-sphere representation. For wavevector geometry, read the printed $x,y,z$ axes as $\alpha,\beta,\gamma$, rather than as physical positions.

### 5.4 A short numerical check

With $\lambda=0.6\,\mu\mathrm m$, choose $\alpha=0.3$ and $\beta=0.4$. The component has $\theta=30^\circ$ and retains its magnitude under propagation. In contrast, $\alpha=1.2$, $\beta=0$ gives $\kappa=k\sqrt{0.44}$; after a distance $1/\kappa$, its **amplitude** falls to $1/e$ and its relative intensity to $1/e^2$.

In [ ]:
wavelength = 0.6  # micrometres; frequencies are then cycles/micrometre
g = propagation.spectrum_geometry(0.3/wavelength, 0.4/wavelength, wavelength)
print(f"Propagating: theta = {float(g['theta_deg']):.1f} deg, gamma = {g['gamma']:.6f}")

ev = propagation.spectrum_geometry(1.2/wavelength, 0, wavelength)
decay_length = 1/float(ev['kz'].imag)
H = propagation.transfer_function(1.2/wavelength, 0, wavelength, decay_length)
print(f"Evanescent amplitude decay length: {decay_length:.4f} micrometres")
print(f"At that distance: |H| = {abs(H):.6f}, |H|^2 = {abs(H)**2:.6f}")

## 6. Fresnel: approximating the longitudinal wavevector

### 6.1 Expand in transverse frequency

Suppose the important components satisfy $q=\lambda^2(f_x^2+f_y^2)\ll1$. The Taylor expansion is **about $q=0$**, not about $z=0$:

$$\sqrt{1-q}=1-\frac q2-\frac{q^2}{8}-\frac{q^3}{16}-\cdots.$$

Retaining the constant and the term linear in $q$ gives

$$
\begin{aligned}
k_z&\approx k\left(1-\frac q2\right)\\
&=k-\frac{k\lambda^2}{2}(f_x^2+f_y^2)\\
&=k-\pi\lambda(f_x^2+f_y^2).
\end{aligned}
$$

Hence the Fresnel transfer function is

$$
\boxed{H_{\mathrm F}=e^{ikz}\exp[-i\pi\lambda z(f_x^2+f_y^2)].}
$$

The first factor is a common carrier phase. The quadratic frequency-dependent phase changes the relative phases between components and can therefore change the reconstructed intensity pattern.

### 6.2 The same approximation in geometry and in real space

The sphere $k_z=\sqrt{k^2-k_x^2-k_y^2}$ becomes a paraboloid near its pole:

$$k_z\approx k-\frac{k_x^2+k_y^2}{2k}.$$

Equivalently, $k_z=k\cos\theta\approx k(1-\theta^2/2)$ for small $\theta$, with $q=\sin^2\theta\approx\theta^2$.

To connect this to the slowly varying envelope, substitute $U=\psi e^{ikz}$ into Helmholtz:

$$
\partial_z^2\psi+2ik\partial_z\psi+\nabla_\perp^2\psi=0.
$$

Neglecting $\partial_z^2\psi$ compared with $2ik\partial_z\psi$ gives the paraxial equation. Its transverse transform obeys

$$
2ik\,\partial_z\widetilde\psi-4\pi^2(f_x^2+f_y^2)\widetilde\psi=0
\quad\Longrightarrow\quad
\widetilde\psi(z)=\widetilde\psi(0)e^{-i\pi\lambda z(f_x^2+f_y^2)}.
$$

Restoring $e^{ikz}$ produces the same $H_{\mathrm F}$.

The quadratic formula has magnitude one at every real frequency. It therefore does **not** reproduce evanescent decay; applying it far outside the paraxial band is not a physical extension of Fresnel theory.

<img src="figures/scalar_wave_propagation_source/image8.gif" width="800" alt="Original animation comparing the exact sphere with the Fresnel paraboloid. Its positive normalized error is $(k_{z,\mathrm F}-k_{z,\mathrm{AS}})/k$. The next section uses the opposite subtraction, so its phase error is negative. The physical error annotation uses the source animation’s chosen wavelength; the normalized curves are general." />

Original animation comparing the exact sphere with the Fresnel paraboloid. Its positive normalized error is $(k_{z,\mathrm F}-k_{z,\mathrm{AS}})/k$. The next section uses the opposite subtraction, so its phase error is negative. The physical error annotation uses the source animation’s chosen wavelength; the normalized curves are general.

## 7. The error is an accumulated phase error

### 7.1 Derive the error before judging the approximation

For propagating components, define exact phase minus Fresnel phase:

$$
\begin{aligned}
\Delta\phi
&=\arg_{\mathrm{unwrapped}}H_{\mathrm{AS}}-
\arg_{\mathrm{unwrapped}}H_{\mathrm F}\\
&=kz\left[\sqrt{1-q}-1+\frac q2\right]\\
&=-kz\left(\frac{q^2}{8}+\frac{q^3}{16}+\cdots\right).
\end{aligned}
$$

This is an unwrapped accumulated phase difference, not a principal angle limited to $[-\pi,\pi]$. An equivalent exact expression avoids subtracting nearly equal numbers when $q$ is small:

$$
\Delta\phi=-\frac{kz\,q^2}{2(1+\sqrt{1-q})^2}.
$$

Keeping the leading term and restoring physical frequencies gives

$$
\boxed{\Delta\phi\approx-\frac{kz}{8}q^2
=-\frac{\pi z\lambda^3}{4}(f_x^2+f_y^2)^2.}
$$

The angular representation follows from $q=\sin^2\theta$:

$$
\begin{aligned}
\Delta\phi
&=kz\left[\cos\theta-1+\frac{\sin^2\theta}{2}\right]\\
&=-\frac{kz}{2}(1-\cos\theta)^2\\
&\approx-\frac{kz}{2}\left(\frac{\theta^2}{2}\right)^2
=-\frac{kz}{8}\theta^4.
\end{aligned}
$$

Angles in these expansions are in radians. The fourth power explains why the outer part of a wide angular spectrum is especially sensitive.

### 7.2 Phase error becomes field error through the input spectrum

$$
\frac{H_{\mathrm{AS}}}{H_{\mathrm F}}=e^{i\Delta\phi},
\qquad
H_{\mathrm{AS}}-H_{\mathrm F}=H_{\mathrm F}(e^{i\Delta\phi}-1).
$$

For a propagating component, both transfer magnitudes are one, so

$$
|H_{\mathrm{AS}}-H_{\mathrm F}|
=2\left|\sin\frac{\Delta\phi}{2}\right|
\approx|\Delta\phi|\quad (|\Delta\phi|\ll1).
$$

The field error is weighted by the spectrum actually present:

$$
\Delta U=\mathcal F^{-1}\{\widetilde U_{\mathrm{ref}}(H_{\mathrm{AS}}-H_{\mathrm F})\},
\qquad
\|\Delta U\|_2^2=\iint |\widetilde U_{\mathrm{ref}}|^2|H_{\mathrm{AS}}-H_{\mathrm F}|^2\,df_x\,df_y.
$$

The second equality is Parseval's relation for finite-power fields. A large transfer error where $\widetilde U_{\mathrm{ref}}$ is negligible may have little effect. A common phase offset leaves intensity unchanged; frequency-dependent errors can change interference, although a single plane wave still has uniform intensity.

### 7.3 A useful criterion, with its comparison stated

If the significant spectrum is confined to transverse radius $f_{\max}$, a small leading phase error requires

$$
\frac{\pi z\lambda^3f_{\max}^4}{4}\ll1,
\qquad\text{or approximately}\qquad
\frac{kz\theta_{\max}^4}{8}\ll1.
$$

At a **fixed input spectrum**, the error accumulates linearly with $z$. A different statement arises when expanding the real-space distance $R=\sqrt{z^2+\rho^2}$ at fixed transverse separation $\rho$:

$$
kR=kz+\frac{k\rho^2}{2z}-\frac{k\rho^4}{8z^3}+\cdots.
$$

That neglected term decreases with $z$ at fixed $\rho$. The two criteria hold different quantities fixed; “Fresnel improves with distance” is incomplete without that distinction. Fresnel and Fraunhofer are also different approximations.

### 7.4 Explore the approximation

The horizontal coordinate below is $s=\lambda\sqrt{f_x^2+f_y^2}=\sqrt q$. Compare the exact longitudinal component, the transfer magnitude, and the accumulated phase error. Increase $z/\lambda$ while keeping $s$ fixed. At $z=0$, both transfer functions are exactly one.

Because both axes use wavelength units, changing wavelength alone at fixed $z/\lambda$ leaves the normalized curves unchanged; it changes the physical distance and frequency scale.

In [ ]:
transfer_lesson = show_scalar_propagation('transfer')

In [ ]:
# One angular component: exact and leading phase error.
wavelength, distance, theta_deg = 0.6, 20.0, 10.0  # micrometres, micrometres, degrees
fx = np.sin(np.deg2rad(theta_deg))/wavelength
error = propagation.fresnel_phase_error(fx, 0, wavelength, distance)
print(f"Exact phase error:   {float(error['exact']):.6f} rad")
print(f"Leading estimate:    {float(error['leading']):.6f} rad")
print(f"Transfer mismatch:   {2*abs(np.sin(error['exact']/2)):.6f}")

## 8. The same propagation in real space

### 8.1 Linearity, shift invariance, and convolution

Define the field propagation operator

$$\mathcal P_z[U_{\mathrm{ref}}]=\mathcal F^{-1}\{H_z\mathcal F[U_{\mathrm{ref}}]\}.$$

Because the Fourier transforms and multiplication by a prescribed $H_z$ are linear,

$$\mathcal P_z[aU_{\mathrm{ref}}+bV_0]=a\mathcal P_z[U_{\mathrm{ref}}]+b\mathcal P_z[V_0].$$

A transverse shift $T_{\mathbf a}U_{\mathrm{ref}}=U_{\mathrm{ref}}(x-a_x,y-a_y)$ changes its spectrum by $e^{-i2\pi(f_xa_x+f_ya_y)}$. This factor commutes with $H_z$, so

$$\mathcal P_z[T_{\mathbf a}U_{\mathrm{ref}}]=T_{\mathbf a}\mathcal P_z[U_{\mathrm{ref}}].$$

Thus propagation in this uniform, unbounded transverse medium is **linear and shift invariant**. Linearity alone would not guarantee shift invariance; a fixed lens or mask can break the latter.

Let $h_z=\mathcal F^{-1}\{H_z\}$ be the response to a point boundary field. The convolution theorem gives

$$
\boxed{U(x,y,z)=(U_{\mathrm{ref}}*h_z)(x,y)
=\iint U_{\mathrm{ref}}(\xi,\eta)h_z(x-\xi,y-\eta)\,d\xi\,d\eta.}
$$

We now add contributions from source-plane positions instead of from Fourier components. Both representations propagate the **complex field**; taking $|U|^2$ is a subsequent nonlinear operation.

### 8.2 The exact Rayleigh–Sommerfeld kernel

For $z>0$, let $R=\sqrt{x^2+y^2+z^2}$. The outgoing spherical Green-function factor is $e^{ikR}/R$. The exact Dirichlet half-space propagation kernel is its normal derivative:

$$
\begin{aligned}
h_{\mathrm{RS}}(x,y;z)
&=-\frac{1}{2\pi}\frac{\partial}{\partial z}\frac{e^{ikR}}R\\
&=-\frac{1}{2\pi}\frac zR\,e^{ikR}
\left(\frac{ik}R-\frac1{R^2}\right)\\
&=\boxed{\frac{z\,e^{ikR}}{2\pi R^3}(1-ikR).}
\end{aligned}
$$

The derivative supplies both the angular weighting and the near-field term. The bare spherical factor $e^{ikR}/R$ alone is not the plane-to-plane propagation kernel.

The outgoing Weyl identity, with the branch chosen in Section 5, gives

$$
\mathcal F_{x,y}\!\left\{\frac{e^{ikR}}R\right\}
=\frac{2\pi i}{k_z}e^{ik_z z}.
$$

Consequently,

$$
\mathcal F_{x,y}\{h_{\mathrm{RS}}\}
=-\frac1{2\pi}\partial_z
\left(\frac{2\pi i}{k_z}e^{ik_z z}\right)
=\boxed{e^{ik_z z}=H_{\mathrm{AS}}.}
$$

The Green-function transform is understood with the outgoing limiting prescription; the apparent singularity at grazing disappears in the differentiated propagation kernel. This establishes the ASM–RS connection: they are Fourier-space and real-space forms of the **same scalar boundary propagator**, including evanescent content. The full Green-identity boundary derivation can be developed in a later notebook. [TU Delft's scalar diffraction derivation](https://qiweb.tudelft.nl/aoi/scalardiffractiontheory/scalardiffractiontheory/) develops the normal-derivative construction; its subsequent large-distance simplification should not be confused with the exact kernel above.

For $kR\gg1$, the term 1 in $(1-ikR)$ is small, giving the familiar approximation

$$h_{\mathrm{RS}}\approx\frac{z}{i\lambda R^2}e^{ikR}.$$

### 8.3 Fresnel in the same language

Taking the inverse transform of the quadratic $H_{\mathrm F}$ yields, for $z>0$,

$$
h_{\mathrm F}(x,y;z)=\frac{e^{ikz}}{i\lambda z}
\exp\!\left[\frac{i\pi}{\lambda z}(x^2+y^2)\right].
$$

Therefore

$$
U_{\mathrm F}(x,y,z)=\frac{e^{ikz}}{i\lambda z}
\iint U_{\mathrm{ref}}(\xi,\eta)
\exp\!\left[\frac{i\pi}{\lambda z}
\bigl((x-\xi)^2+(y-\eta)^2\bigr)\right]d\xi\,d\eta.
$$

This is the same Fresnel approximation derived in frequency space, now written as contributions from points on the input plane.

## 9. A first FFT calculation

The discrete calculation mirrors the derivation:

```python
A0 = np.fft.fft2(U0)
Az = A0 * H
Uz = np.fft.ifft2(Az)
```

NumPy's FFT returns an unscaled discrete spectrum; the forward and inverse normalizations cancel in this propagation pair. To approximate the continuous $\widetilde U_{\mathrm{ref}}$ itself, the sample-area factor $\Delta x\Delta y$ and coordinate-origin phase must also be accounted for. The frequency grid must match the FFT ordering; no `fftshift` is needed for the multiplication.

This demonstration propagates a specified **Gaussian boundary field** using both exact ASM multipliers and Fresnel multipliers. It does not assume that a usual paraxial Gaussian formula is an exact Helmholtz solution. Change the independent widths or the distance in the code.

The computation uses a finite periodic grid with no padding or band limiting. The displayed central region is smaller than that grid. A physical isolated-aperture calculation requires checks of sampling, window size, wraparound, and transfer-function sampling; these belong to the later computational ASM notebook.
- The present smooth, well-contained example is a starting point.

In [ ]:
# All lengths are in micrometres. Array indexing is [y, x].
wavelength = 0.6
distance = 8.0
waist_x, waist_y = 1.2, 2.0  # separate 1/e field radii at z=0
nx = ny = 256
dx = dy = 0.25
x = (np.arange(nx)-nx//2)*dx
y = (np.arange(ny)-ny//2)*dy
xx, yy = np.meshgrid(x, y)
U0 = np.exp(-(xx/waist_x)**2-(yy/waist_y)**2).astype(complex)

asm = propagation.propagate_field(U0, dx, dy, wavelength, distance, 'asm')
fresnel = propagation.propagate_field(U0, dx, dy, wavelength, distance, 'fresnel')
U_AS, U_F = asm['field'], fresnel['field']
relative_field_error = np.linalg.norm(U_AS-U_F)/np.linalg.norm(U_AS)
print(f"Relative complex-field difference: {relative_field_error:.4%}")
print(f"Computed window: {nx*dx:g} x {ny*dy:g} micrometres; displayed: -8 to +8")

fig = Figure(figsize=(10, 8))
FigureCanvasAgg(fig)
axes = fig.subplots(2, 2).ravel()
fig.subplots_adjust(left=.08, right=.94, bottom=.09, top=.94, wspace=.42, hspace=.35)
extent = [x[0]-dx/2, x[-1]+dx/2, y[0]-dy/2, y[-1]+dy/2]
for ax, field, title in zip(axes[:3], (U0, U_AS, U_F),
                           ('Input |U0|²', 'ASM |U(z)|²', 'Fresnel |U(z)|²')):
    im = ax.imshow(abs(field)**2, origin='lower', extent=extent,
                   cmap='magma', vmin=0, vmax=1, interpolation='nearest')
    ax.set(xlim=(-8, 8), ylim=(-8, 8), xlabel='x (µm)', ylabel='y (µm)', title=title)
    fig.colorbar(im, ax=ax, shrink=.82, pad=.04, label='Relative intensity')
axes[3].plot(x, abs(U0[ny//2])**2, color='0.6', label='Input')
axes[3].plot(x, abs(U_AS[ny//2])**2, label='ASM')
axes[3].plot(x, abs(U_F[ny//2])**2, '--', label='Fresnel')
axes[3].set(xlim=(-8, 8), ylim=(0, 1.05), xlabel='x (µm)',
            ylabel='Relative intensity', title='Profile at y=0')
axes[3].legend(fontsize=8)
show_figure(fig)

The intensity maps share the same fixed unit-input reference. They are not independently normalized at each plane. The wider profile and reduced peak show spreading; the relative complex-field difference also captures phase differences that an intensity-only comparison could miss.

An immediate consistency check is $z=0$: both methods must return the input. A second check is a single discrete Fourier mode: its propagated field must equal that mode times its own $H$. The shared numerical module has tests for both, evanescent decay, and the analytic paraxial Gaussian limit.

## Optional A. Other representations retained from the document

### A.1 General mode expansions and beams

Linearity allows $U=\sum_p c_p\psi_p$ whenever the $\psi_p$ solve the same linear equation. It does not make an arbitrary collection complete or orthogonal; those properties depend on the basis, domain, and boundary conditions. Plane waves, spherical waves, and cylindrical waves organize different geometries. Standard Gaussian, Hermite–Gaussian, and Laguerre–Gaussian beam formulas belong to paraxial theory.

The angular spectrum also explains beam families. Directions on a cone share $k_z$ and transverse magnitude $k_\perp$. Their azimuthal superposition is

$$
\frac1{2\pi}\int_0^{2\pi}
e^{im\varphi_k}
e^{i k_\perp\rho\cos(\varphi_k-\varphi)}e^{ik_z z}\,d\varphi_k
=i^m e^{im\varphi}J_m(k_\perp\rho)e^{ik_z z},
$$

where $k_\perp^2+k_z^2=k^2$ and $m$ is an integer. For $m=0$, this is the rotationally symmetric Bessel beam. The ideal unlimited Bessel beam is an exact scalar solution with infinite transverse power; an experimentally finite aperture changes its propagation range.

<img src="figures/scalar_wave_propagation_source/image11.jpeg" width="475" alt="Source cone-spectrum representation. Its weight $e^{im\varphi_k}$ adds azimuthal phase winding; all directions share the same longitudinal wavevector." />

Source cone-spectrum representation. Its weight $e^{im\varphi_k}$ adds azimuthal phase winding; all directions share the same longitudinal wavevector.

### A.2 Angular coordinates also change the integration measure

For the propagating part,

$$
f_x=\frac{\sin\theta\cos\varphi}{\lambda},\qquad
f_y=\frac{\sin\theta\sin\varphi}{\lambda},
\qquad
df_x\,df_y=\frac{\sin\theta\cos\theta}{\lambda^2}\,d\theta\,d\varphi.
$$

Consequently,

$$
U_{\mathrm{prop}}(\mathbf r)=
\int_0^{2\pi}\!\int_0^{\pi/2}
\widetilde U\!\left(\frac{\sin\theta\cos\varphi}{\lambda},
\frac{\sin\theta\sin\varphi}{\lambda};0\right)
e^{ik\hat{\mathbf s}\cdot\mathbf r}
\frac{\sin\theta\cos\theta}{\lambda^2}\,d\theta\,d\varphi.
$$

Likewise $df_xdf_y=d\alpha d\beta/\lambda^2=dk_xdk_y/(2\pi)^2$. Changing coordinates changes the spectral density as well as the labels. The real-angle integral above excludes evanescent components.

### A.3 Refractive-index-scaled coordinates

If $\lambda_0$ is the vacuum wavelength, then $\lambda=\lambda_0/n$ and $k=nk_0$, with $k_0=2\pi/\lambda_0$. The alternative vector in the source figure is

$$
\mathbf j=(p,q_{\mathrm{fig}},\sqrt{n^2-p^2-q_{\mathrm{fig}}^2})
=n(\alpha,\beta,\gamma),\qquad \mathbf k=k_0\mathbf j.
$$

Thus $p=n\alpha$ and $q_{\mathrm{fig}}=n\beta$. Its symbol $q$ is a coordinate, whereas our main derivation uses $q=\alpha^2+\beta^2$. Keeping the figure's labels requires this distinction.

<img src="figures/scalar_wave_propagation_source/image10.jpeg" width="430" alt="Source index-scaled direction geometry. The vector has length $n$, rather than one; its $p,q$ are not the unit direction cosines." />

Source index-scaled direction geometry. The vector has length $n$, rather than one; its $p,q$ are not the unit direction cosines.

### A.4 A global three-dimensional Fourier shell

For a globally source-free, tempered monochromatic field, transforming all three coordinates gives

$$
(k^2-|\mathbf K|^2)\widetilde U(\mathbf K)=0.
$$

Here $\mathbf K$ is the real 3D Fourier variable. The spectrum is supported on $|\mathbf K|=k$, and can be written distributionally as

$$\widetilde U(\mathbf K)=B(\mathbf K)\delta(k^2-|\mathbf K|^2),$$

with normalization absorbed into the shell weight $B$. Away from grazing, putting $\kappa=\sqrt{k^2-K_x^2-K_y^2}>0$ gives

$$
\delta(k^2-K_x^2-K_y^2-K_z^2)
=\frac{\delta(K_z-\kappa)+\delta(K_z+\kappa)}{2\kappa}.
$$

This displays the two hemispheres and the measure factor that relates their weights to a transverse spectrum.

- The global shell is not interchangeable with an unrestricted boundary spectrum.
- A half-space evanescent field is valid in ASM but grows if continued backward without a source or boundary; it is outside the assumptions of this global real-wavevector representation.
- An outgoing spherical Green function also has a point source, so its full 3D spectrum is not solely a delta shell.

### A.5 Phase zeros and the statistical-optics extension

At $U=0$, phase is undefined. A phase singularity with nonzero winding around a zero is an optical vortex; an arbitrary field zero need not be a vortex.

The document's random-field representation can be retained as

$$U(\mathbf r)=U_{\mathrm{ref}}(\mathbf r)e^{\psi(\mathbf r)},
\qquad \psi=\chi+i\varphi.$$

Here $\chi$ is a log-amplitude fluctuation and $\varphi$ a phase fluctuation. Such a logarithmic representation needs care at zeros. Statistical optics then studies quantities such as

$$
\langle U(\mathbf r)\rangle,\qquad
W(\mathbf r_1,\mathbf r_2)=\langle U^*(\mathbf r_1)U(\mathbf r_2)\rangle,
\qquad
D_\varphi(\mathbf r_1,\mathbf r_2)
=\langle[\varphi(\mathbf r_2)-\varphi(\mathbf r_1)]^2\rangle.
$$

The same deterministic ASM operator propagates each realization. In spectral form, if

$$W_{\widetilde U}(\mathbf f_1,\mathbf f_2;0)=\langle \widetilde U_{\mathrm{ref}}^*(\mathbf f_1)\widetilde U_{\mathrm{ref}}(\mathbf f_2)\rangle,$$

then

$$
W_{\widetilde U}(\mathbf f_1,\mathbf f_2;z)
=H^*(\mathbf f_1;z)H(\mathbf f_2;z)W_{\widetilde U}(\mathbf f_1,\mathbf f_2;0).
$$

This provides a bridge to coherence theory. The main derivation in this notebook needs no random-medium or partial-coherence model.

### A.6 Two additional source illustrations

These original figures are kept for context. Their conventions differ from the main propagation diagrams, as noted below.

<img src="figures/scalar_wave_propagation_source/image1.png" width="500" alt="Source comparison of longitudinal and transverse physical oscillations. Light in a homogeneous isotropic medium is transverse in the vector description; a scalar field plot is a graph of a component, not a material displacement or a wavy ray trajectory." />

Source comparison of longitudinal and transverse physical oscillations. Light in a homogeneous isotropic medium is transverse in the vector description; a scalar field plot is a graph of a component, not a material displacement or a wavy ray trajectory.

<img src="figures/scalar_wave_propagation_source/image12.png" width="700" alt="Source sinusoidal field and directional-derivative illustration. It uses $x$ to the right, $z$ downward, and $m(x,z)=\cos(k_xx-k_zz)$; its arrow therefore has components $(k_x,-k_z)$ in those coordinates. Our main convention is $e^{i(k_xx+k_zz)}$. The companion plot is a directional-derivative norm, not a propagated intensity map." />

Source sinusoidal field and directional-derivative illustration. It uses $x$ to the right, $z$ downward, and $m(x,z)=\cos(k_xx-k_zz)$; its arrow therefore has components $(k_x,-k_z)$ in those coordinates. Our main convention is $e^{i(k_xx+k_zz)}$. The companion plot is a directional-derivative norm, not a propagated intensity map.

## Checkpoint

1. Why is a 2D Fourier transform sufficient to start the boundary propagation problem?
2. What extra condition selects $\widetilde U(z)=\widetilde U(0)e^{ik_z z}$ from the second-order ODE?
3. For $\alpha=0.6$, $\beta=0$, what are $\gamma$ and the polar angle? What changes if $\alpha=1.2$?
4. What is being approximated in Fresnel theory: the transverse Fourier decomposition or the longitudinal wavevector?
5. Why can a small error in $k_z$ matter after a long distance?
6. Does every phase error necessarily change intensity?
7. Why is $e^{ikR}/R$ alone not the RS plane-to-plane kernel?
8. Can exact ASM formulas produce inaccurate numerical results?

<details>
<summary><b>Answers</b></summary>

1. The prescribed field lives on the two-coordinate plane $z=0$. The transformed Helmholtz equation determines how each component evolves with $z$.
2. The outgoing/decaying condition into $z>0$, together with bounded continuation at grazing. Boundary values alone do not choose between forward and backward components.
3. $\gamma=\sqrt{1-0.6^2}=0.8$ and $\theta=\arcsin(0.6)\approx36.87^\circ$. At $\alpha=1.2$, $\gamma=i\sqrt{0.44}$: the field decays along $z$ and has no ordinary real polar angle.
4. The longitudinal component: $k_z=k\sqrt{1-q}$ is replaced by $k(1-q/2)$. The transverse Fourier basis is unchanged.
5. The accumulated phase difference is $(k_{z,\mathrm{AS}}-k_{z,\mathrm F})z$. For a fixed spectrum it grows linearly with distance.
6. No. A common phase does not change $|U|^2$; a single plane wave stays uniform. Different errors across interfering components can change the pattern.
7. The boundary propagator is a normal derivative of the spherical Green-function factor, including the angular and near-field factors. Its transverse Fourier transform equals $H_{\mathrm{AS}}$.
8. Yes. A finite FFT grid can introduce aliasing, wraparound, truncation, and inadequate transfer-function sampling even when the continuous transfer formula is exact.

</details>

## Source, retained material, and references

**Source document:** `Scalar_wave_propagation.docx`, in `Mat_codes/Docs/Classical_Coherence_theory/All_Word`. The Word file is unchanged. All twelve unique embedded figures are retained under `figures/scalar_wave_propagation_source/`; repeated uses of the same image are consolidated. Both GIFs retain all 88 frames. The original EMF is retained with a PNG display conversion. [The figure inventory](figures/scalar_wave_propagation_source/inventory.json) records source locations, hashes, and notation guidance.

**Editorial and mathematical adjustments:** the $k_z$ chain now has consistent signs and parentheses; the Fresnel expansion is about small transverse frequency; the angular and frequency error criteria retain the required fourth powers. Field normalization is distinguished from electric-field units. The global Fourier-shell assumptions and the exact RS derivative kernel are explicit. The document's opening discussion of linear systems is completed with the shift-invariance and convolution steps. The numerical examples and interactive views are additions.

For the derivation and subsequent work:

[TU Delft, diffraction and angular spectrum](https://interactivetextbooks.tudelft.nl/interactive-optics/content/Chap6_Diffraction/DiffractiveOptics_2022_01Clean.html): propagation, approximations, and the relation to RS. [TU Delft, scalar diffraction theory](https://qiweb.tudelft.nl/aoi/scalardiffractiontheory/scalardiffractiontheory/): Green identities and boundary-integral construction. [W. C. Chew, Purdue Lecture 6](https://engineering.purdue.edu/wcchew/ece604f20/Lecture%20Notes/Lect6.pdf): time-harmonic phasors and complex power, also listed in the document. [Kozacki, Falaggis, and Kujawinska, Applied Optics 51, 7080–7088 (2012)](https://opg.optica.org/ao/abstract.cfm?uri=ao-51-29-7080): high-NA numerical ASM considerations. An exact continuous formulation still requires a careful discrete implementation.

Additional links retained from the document: [Jasper Landa thesis](https://pure.tue.nl/ws/portalfiles/portal/93321765/Thesis_Jasper_Landa.pdf), [angular-spectrum figure in Optical Tweezers](http://opticaltweezers.org/chapter-4/figure-4-4-angular-spectrum-representation/), [Photonics 11(9), 871](https://www.mdpi.com/2304-6732/11/9/871), and [Purdue imaging-systems notes](https://engineering.purdue.edu/~ece438/lecture_notes/legacy_notes/module_2/2.1_systems_spect_analysis/2.1.3_imaging_systems.pdf).

For the next notebook, the natural continuation is the numerical ASM problem: sampling, finite windows, padding, evanescent filtering, band limits, and convergence checks against an independent representation.

## Selected freely accessible reading

Optional reading: start with the suggested passage. Freely accessible works are not necessarily in the public domain.

- **K. Matsushima and T. Shimobaba. [Band-Limited Angular Spectrum Method for Numerical Simulation of Free-Space Propagation in Far and Near Fields](https://kansai-u.repo.nii.ac.jp/record/11127/files/KU-1100-20091015-50.pdf).** Optics Express 17, 19662–19673 (2009); DOI: 10.1364/OE.17.019662. **Access:** Free institutional-repository full text; copyright retained. **Read for:** Read the sampling discussion; this notebook does not implement that algorithm.

- **M. Mansuripur. [A Tutorial on the Classical Theories of Electromagnetic Scattering and Diffraction](https://arxiv.org/abs/2009.06706).** Nanophotonics 10, 315–342 (2021); arXiv version submitted 2020. **Access:** Free arXiv manuscript; not public domain. **Read for:** Optional advanced reading: scalar diffraction and its electromagnetic assumptions.

[Source catalogue and access notes](reading_sources.json).
